# Notebook guiado — K-Means Clustering desde cero

**RA1 · Programación de Inteligencia Artificial**  
**Notebook de refuerzo · Google Colab (Versión K-Means)**

En este notebook aprenderemos el algoritmo K-Means de forma visual y progresiva.

> **El objetivo de K-Means es encontrar agrupaciones naturales (clusters) en los datos minimizando la distancia de los puntos a su centro de gravedad, sin conocer ninguna etiqueta o respuesta previa.**


## 1. K-Means manual paso a paso

Primero, veamos cómo el algoritmo busca los centros de gravedad de forma iterativa sin usar scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Creamos puntos agrupados visualmente, pero sin decirles a qué grupo pertenecen
X_demo, _ = make_blobs(n_samples=150, centers=3, cluster_std=0.8, random_state=42)

df_demo = pd.DataFrame({"x1": X_demo[:, 0], "x2": X_demo[:, 1]})

plt.figure(figsize=(8, 6))
plt.scatter(df_demo["x1"], df_demo["x2"], c='gray', s=50)
plt.title("Datos originales (sin agrupar)")
plt.show()

In [ ]:
# 1. Inicialización (k=3)
k = 3
centroides = X_demo[np.random.choice(X_demo.shape[0], k, replace=False)]

# Bucle de optimización (simplificado a 5 iteraciones)
for i in range(5):
    # 2. Asignación: distancia euclídea
    distancias = np.linalg.norm(X_demo[:, np.newaxis] - centroides, axis=2)
    grupos = np.argmin(distancias, axis=1)
    
    # 3. Actualización: mover centroide a la media
    nuevos_centroides = np.array([X_demo[grupos == j].mean(axis=0) for j in range(k)])
    centroides = nuevos_centroides

plt.figure(figsize=(8, 6))
plt.scatter(X_demo[:, 0], X_demo[:, 1], c=grupos, cmap='viridis', s=50)
plt.scatter(centroides[:, 0], centroides[:, 1], c='red', marker='X', s=200, label='Centroides')
plt.title("K-Means manual tras 5 iteraciones")
plt.legend()
plt.show()

## 2. K-Means real con el dataset Iris

Vamos a aplicarlo a flores reales, pero ocultando las especies al algoritmo.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

iris = load_iris(as_frame=True)
df_iris = iris.frame.copy()

# Variables independientes
X = df_iris[['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']]

# Pipeline con escalado
modelo_kmeans = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=3, random_state=42, n_init=10))
])

grupos_descubiertos = modelo_kmeans.fit_predict(X)
print("Asignaciones listas.")

## 3. Evaluación y Método del Codo

Evaluamos el modelo midiendo la Inercia y el Coeficiente de Silueta, y buscamos el K óptimo.

In [ ]:
from sklearn.metrics import silhouette_score

modelo_puro = modelo_kmeans.named_steps["kmeans"]
X_escalado = modelo_kmeans.named_steps["scaler"].transform(X)

print(f"Inercia del modelo: {modelo_puro.inertia_:.2f}")
print(f"Puntuación de Silueta: {silhouette_score(X_escalado, grupos_descubiertos):.3f}")

# Método del Codo
inercias = []
rango_k = range(1, 11)

for k in rango_k:
    pipe_temp = Pipeline([
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(n_clusters=k, random_state=42, n_init=10))
    ])
    pipe_temp.fit(X)
    inercias.append(pipe_temp.named_steps["kmeans"].inertia_)

plt.figure(figsize=(8, 5))
plt.plot(rango_k, inercias, marker='o')
plt.axvline(x=3, color='r', linestyle='--')
plt.title("Método del Codo para elegir k")
plt.show()

## 4. Fronteras de decisión

Las fronteras en K-Means forman celdas de Voronoi.

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

X_2d = df_iris[["petal length (cm)", "petal width (cm)"]]
modelo_2d = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=3, random_state=42, n_init=10))
])
modelo_2d.fit(X_2d)

plt.figure(figsize=(9, 6))
DecisionBoundaryDisplay.from_estimator(
    modelo_2d, X_2d, response_method="predict", alpha=0.4, cmap="viridis"
)
plt.scatter(X_2d["petal length (cm)"], X_2d["petal width (cm)"], 
            c=modelo_2d.predict(X_2d), edgecolor='k', s=50, cmap="viridis")
plt.title("Fronteras de decisión (Voronoi) en K-Means")
plt.show()

## 5. Mini reto final

1. Importa un dataset limpio (puedes usar `load_wine` o `load_breast_cancer` de `sklearn.datasets`).
2. Escala las variables numéricas.
3. Programa un bucle for que evalúe el *Silhouette Score* para valores de `k` entre 2 y 8.
4. Entrena el modelo final con el `k` que haya sacado mejor puntuación.

In [ ]:
# TU CÓDIGO AQUÍ